# 02 — Feature Extraction

Runs the two frozen backbones once over the corpus and caches their outputs:

* **EfficientNet-B4** over face crops → `(30, 1792)` per clip
* **Wav2Vec 2.0 Base** over the 16 kHz waveform → `(30, 768)` per clip

Why cache instead of extracting on the fly? Because the backbones are *frozen*, their
outputs never change across epochs or ablations. Re-running them every step would waste
most of your GPU time, and — more importantly — caching holds the input representation
**identical across every ablation**, so any difference we measure later is attributable
to the training objective rather than to backbone drift. That is a methodological
requirement, not just an optimisation.

**Time:** roughly 3–6 hours for ~20k clips on an A5000, dominated by face detection and
video decoding. It checkpoints and resumes, so interruptions are safe.

## 2.1 Config

In [ ]:
import json, os
from pathlib import Path
import numpy as np
import torch

DATASET = "lavdf"        # "lavdf"  = pass 1, primary training corpus
                         # "fakeavceleb" = pass 2, cross-dataset test corpus
DATA_DIR = Path("../data")
PATHS = {
    "lavdf":       (DATA_DIR / "raw" / "LAV-DF",           DATA_DIR / "feats" / "lavdf"),
    "fakeavceleb": (DATA_DIR / "raw" / "FakeAVCeleb_v1.2", DATA_DIR / "feats" / "fakeavceleb"),
}
RAW_ROOT, FEAT_ROOT = PATHS[DATASET]
RUNS_ROOT = DATA_DIR / "runs"            # checkpoints and logs (notebooks 03 and 04)
SEQ_LEN   = 30        # T aligned windows per clip
IMG_SIZE  = 224
FACE_PAD  = 0.20      # 20% padding to retain periocular context

manifest_path = FEAT_ROOT / "manifest.json"
if not manifest_path.exists():
    raise FileNotFoundError(
        f"{manifest_path.resolve()} not found. Run notebook 01 with DATASET = '{DATASET}' first.")
manifest = json.loads(manifest_path.read_text())
rows = manifest["rows"]
print(f"dataset: {DATASET}   dataset_kind in manifest: {manifest.get('dataset_kind')}")
print(f"{len(rows)} clips to process")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## 2.2 Build the encoders

Both are frozen. `timm` gives EfficientNet-B4 with `num_classes=0`, i.e. global-average-
pooled features (1792-d) rather than logits.

In [ ]:
import timm
import torchaudio

visual_encoder = timm.create_model("efficientnet_b4", pretrained=True, num_classes=0)
visual_encoder.eval().to(device)
for p in visual_encoder.parameters():
    p.requires_grad_(False)

bundle = torchaudio.pipelines.WAV2VEC2_BASE
audio_encoder = bundle.get_model().eval().to(device)
for p in audio_encoder.parameters():
    p.requires_grad_(False)
SR = bundle.sample_rate     # 16000

print("EfficientNet-B4 -> 1792-d | Wav2Vec2 Base -> 768-d | sr =", SR)

## 2.3 Face detector

Two options. **MTCNN (facenet-pytorch) is GPU-accelerated and ~5-10x faster** than dlib's
HOG detector; on 20k clips that difference is hours. Use it unless you have a reason not
to.

In [ ]:
USE_MTCNN = True   # False -> dlib HOG via face_recognition

if USE_MTCNN:
    from facenet_pytorch import MTCNN
    mtcnn = MTCNN(keep_all=False, device=device, post_process=False)

    def detect_face_box(rgb):
        """rgb: HxWx3 uint8. Returns (top,right,bottom,left) or None."""
        boxes, _ = mtcnn.detect(rgb)
        if boxes is None or len(boxes) == 0:
            return None
        x1, y1, x2, y2 = boxes[0]
        return int(y1), int(x2), int(y2), int(x1)
else:
    import face_recognition

    def detect_face_box(rgb):
        locs = face_recognition.face_locations(rgb, model="hog")
        return locs[0] if locs else None

print("face detector:", "MTCNN (GPU)" if USE_MTCNN else "dlib HOG (CPU)")

## 2.4 Extraction functions

In [ ]:
import cv2

IMNET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMNET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


def extract_visual(video_path):
    """Decode, uniformly sample SEQ_LEN frames, crop faces, encode. -> (T,1792) or None"""
    cap = cv2.VideoCapture(str(video_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        return None

    want = set(np.linspace(0, total - 1, SEQ_LEN).astype(int).tolist())
    frames, i = [], 0
    while True:
        ok, fr = cap.read()
        if not ok:
            break
        if i in want:
            frames.append(fr)
        i += 1
    cap.release()
    if not frames:
        return None
    # pad if the decoder gave us fewer than requested
    while len(frames) < SEQ_LEN:
        frames.append(frames[-1])
    frames = frames[:SEQ_LEN]

    crops = []
    for fr in frames:
        rgb = cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)
        box = detect_face_box(rgb)
        if box is not None:
            t, r, b, l = box
            ph, pw = int(FACE_PAD * (b - t)), int(FACE_PAD * (r - l))
            t, b = max(0, t - ph), min(rgb.shape[0], b + ph)
            l, r = max(0, l - pw), min(rgb.shape[1], r + pw)
            face = rgb[t:b, l:r]
            if face.size == 0:
                face = rgb
        else:
            face = rgb          # fall back to full frame
        crops.append(cv2.resize(face, (IMG_SIZE, IMG_SIZE)))

    x = torch.from_numpy(np.stack(crops)).permute(0, 3, 1, 2).float() / 255.0
    x = (x - IMNET_MEAN) / IMNET_STD
    with torch.no_grad():
        return visual_encoder(x.to(device)).cpu()      # (T,1792)


def extract_audio(video_path):
    """Load audio, run Wav2Vec2, pool to SEQ_LEN windows. -> (T,768) or None"""
    try:
        wav, in_sr = torchaudio.load(str(video_path))
    except Exception:
        return None
    if wav.numel() == 0:
        return None
    if wav.size(0) > 1:
        wav = wav.mean(0, keepdim=True)
    if in_sr != SR:
        wav = torchaudio.functional.resample(wav, in_sr, SR)

    with torch.no_grad():
        feats, _ = audio_encoder.extract_features(wav.to(device))
        h = feats[-1].squeeze(0).cpu()                  # (frames, 768)
    if h.size(0) == 0:
        return None
    if h.size(0) < SEQ_LEN:
        h = torch.nn.functional.pad(h, (0, 0, 0, SEQ_LEN - h.size(0)))

    # mean-pool into SEQ_LEN contiguous windows, aligned with the video sampling
    idx = np.linspace(0, h.size(0), SEQ_LEN + 1).astype(int)
    pooled = torch.stack([h[idx[i]:max(idx[i] + 1, idx[i+1])].mean(0)
                          for i in range(SEQ_LEN)])
    return pooled                                       # (T,768)

## 2.5 Run it (resumable)

Skips clips already extracted, so you can interrupt and re-run this cell freely.

Clips that fail — no face found, no audio stream, corrupt file — are logged to
`failed.txt` rather than silently dropped. **Check that file afterwards.** A large
failure count concentrated in one category would bias your dataset.

In [ ]:
from tqdm.auto import tqdm

FEAT_ROOT.mkdir(parents=True, exist_ok=True)
failed = []
skipped = 0

for r in tqdm(rows, desc="extracting"):
    stem = r["path"].replace("/", "__").replace(".mp4", "")
    out = FEAT_ROOT / f"{stem}.npz"
    if out.exists():
        skipped += 1
        continue

    vp = RAW_ROOT / r["path"]
    try:
        vf = extract_visual(vp)
        if vf is None:
            failed.append((r["path"], "visual"))
            continue
        af = extract_audio(vp)
        if af is None:
            failed.append((r["path"], "audio"))
            continue
        np.savez_compressed(
            out,
            visual=vf.numpy().astype(np.float32),
            audio=af.numpy().astype(np.float32),
            label=np.int64(r["label"]),
            subject=str(r["subject"]),
            method=str(r["method"]),
        )
    except Exception as e:
        failed.append((r["path"], f"error: {e}"))

print(f"\nskipped (already done): {skipped}")
print(f"failed: {len(failed)}")
(FEAT_ROOT / "failed.txt").write_text(
    "\n".join(f"{p}\t{why}" for p, why in failed))

## 2.6 Verify the cache before you trust it

In [ ]:
import collections

files = sorted(FEAT_ROOT.glob("*.npz"))
print(f"cached clips: {len(files)}")
assert len(files) > 0, "Nothing extracted. Check the failure log."

# shape check on a sample
d = np.load(files[0], allow_pickle=True)
print("visual:", d["visual"].shape, d["visual"].dtype)
print("audio :", d["audio"].shape,  d["audio"].dtype)
assert d["visual"].shape == (SEQ_LEN, 1792), "visual shape wrong"
assert d["audio"].shape  == (SEQ_LEN, 768),  "audio shape wrong"

# sanity: features must not be all-zero or NaN
assert np.isfinite(d["visual"]).all() and np.isfinite(d["audio"]).all(), "NaN/Inf in features"
assert np.abs(d["visual"]).sum() > 0, "visual features are all zero"

# --- did failures bias the class balance? ---
labels = [int(np.load(f, allow_pickle=True)["label"]) for f in files]
cnt = collections.Counter(labels)
print(f"\ncached label balance: REAL={cnt[0]}  FAKE={cnt[1]}  "
      f"({100*cnt[0]/len(labels):.1f}% real)")

fail_rate = len(failed) / max(1, len(rows))
if fail_rate > 0.10:
    print(f"\n!! {100*fail_rate:.1f}% of clips failed extraction. Inspect failed.txt -- "
          "if failures cluster in one category, your dataset is now biased and the "
          "results will be misleading.")
else:
    print(f"\nfailure rate {100*fail_rate:.1f}% -- acceptable.")

print("\nNext: 03_train.ipynb")